# Malicious URL Classification

End-to-end machine learning workflow for classifying URLs as benign or malicious. This notebook reuses the shared project modules so the notebook, batch training script, and Streamlit app all stay in sync.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

from src.evaluate import build_metrics_table
from src.feature_engineering import build_feature_dataframe
from src.preprocessing import analyze_dataset, clean_dataset, load_dataset, save_missing_values_heatmap, split_train_test
from src.train import compare_vectorizers, run_training_pipeline, train_models
from src.utils import ensure_directory

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    if (current / 'data' / 'dataset.csv').exists():
        return current
    if (current.parent / 'data' / 'dataset.csv').exists():
        return current.parent
    return current

PROJECT_ROOT = find_project_root()
CSV_PATH = PROJECT_ROOT / 'data' / 'dataset.csv'
OUTPUTS_DIR = ensure_directory(PROJECT_ROOT / 'outputs')
FIGURES_DIR = ensure_directory(OUTPUTS_DIR / 'figures')
REPORTS_DIR = ensure_directory(OUTPUTS_DIR / 'reports')
METRICS_DIR = ensure_directory(OUTPUTS_DIR / 'metrics')

## 1. Dataset Loading

Load the CSV file and inspect the raw schema before any transformation.

In [ ]:
df = load_dataset(CSV_PATH)
url_column, label_column = analyze_dataset(df)['url_column'], analyze_dataset(df)['label_column']
print('Dataset shape:', df.shape)
print('URL column:', url_column)
print('Label column:', label_column)
display(df.head())
display(df.tail())

## 2. Data Analysis

Compute core dataset statistics and generate the requested visualizations.

In [ ]:
analysis = analyze_dataset(df)
print('Samples:', analysis['samples'])
print('Columns:', analysis['columns'])
print('Unique URLs:', analysis['unique_urls'])
print('Missing values:', analysis['missing_values'])
print('Duplicate rows:', analysis['duplicate_rows'])
print('Label distribution:', analysis['label_distribution'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(x=df[label_column].astype(str), ax=axes[0], palette='viridis')
axes[0].set_title('Class Distribution')
axes[0].set_xlabel('Label')
axes[0].set_ylabel('Count')

url_lengths = df[url_column].astype(str).str.len()
sns.histplot(url_lengths, bins=50, ax=axes[1], color='#1f77b4')
axes[1].set_title('URL Length Distribution')
axes[1].set_xlabel('URL Length')
axes[1].set_ylabel('Frequency')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'dataset_analysis.png', dpi=200, bbox_inches='tight')
plt.show()

save_missing_values_heatmap(df, FIGURES_DIR)
plt.figure(figsize=(6, 4))
df[label_column].astype(str).value_counts().plot.pie(autopct='%1.1f%%', ylabel='', title='Label Distribution Pie Chart')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'label_pie_chart.png', dpi=200, bbox_inches='tight')
plt.show()

## 3. Cleaning

Remove invalid rows, normalize labels, strip whitespace, and deduplicate URLs.

In [ ]:
print('Before cleaning:')
print('  Rows:', len(df))
print('  Missing values:', int(df.isna().sum().sum()))
print('  Duplicate rows:', int(df.duplicated().sum()))

cleaned = clean_dataset(df)
cleaned_df = cleaned.dataframe
print('After cleaning:')
print('  Rows:', len(cleaned_df))
print('  Unique URLs:', cleaned_df[cleaned.url_column].nunique())
print('  Labels:', cleaned_df[cleaned.label_column].value_counts().to_dict())
display(cleaned_df.head())

## 4. Feature Engineering

Generate handcrafted URL features, including length, character counts, entropy, and suspicious word flags.

In [ ]:
feature_sample = build_feature_dataframe(cleaned_df[cleaned.url_column].head(10))
display(feature_sample)
print('Feature columns:', list(feature_sample.columns))

## 5. Vectorization

Compare CountVectorizer and TF-IDF across character and word n-grams.

In [ ]:
X_train, X_test, y_train, y_test = split_train_test(cleaned_df, cleaned.url_column, cleaned.label_column)
vectorizer_sample = X_train.sample(n=min(10000, len(X_train)), random_state=42)
vectorizer_labels = y_train.loc[vectorizer_sample.index]
vectorizer_comparison, best_vectorizer_name = compare_vectorizers(vectorizer_sample, vectorizer_labels, X_test, y_test)
display(vectorizer_comparison)
print('Best vectorizer:', best_vectorizer_name)
vectorizer_comparison.to_csv(METRICS_DIR / 'vectorizer_comparison.csv', index=False)

## 6. Balancing

The class distribution is imbalanced, so the training pipeline applies SMOTE to the sampled training folds when needed.

In [ ]:
before_counts = y_train.value_counts().sort_index()
print('Before balancing:', before_counts.to_dict())
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
before_counts.plot(kind='bar', ax=axes[0], color=['#4c78a8', '#f58518'])
axes[0].set_title('Before SMOTE')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Count')

from imblearn.over_sampling import SMOTE
from src.train import build_vectorizers, build_feature_union

smote_preview = SMOTE(random_state=42, k_neighbors=3)
X_preview = build_feature_union(build_vectorizers()[best_vectorizer_name]).fit_transform(vectorizer_sample)
X_resampled, y_resampled = smote_preview.fit_resample(X_preview, vectorizer_labels)
pd.Series(y_resampled).value_counts().sort_index().plot(kind='bar', ax=axes[1], color=['#4c78a8', '#f58518'])
axes[1].set_title('After SMOTE')
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Count')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'smote_comparison.png', dpi=200, bbox_inches='tight')
plt.show()
print('After balancing:', pd.Series(y_resampled).value_counts().sort_index().to_dict())

## 7. Train/Test Split

Split the cleaned dataset into 80% training and 20% testing with stratification.

In [ ]:
print('Training samples:', len(X_train))
print('Testing samples:', len(X_test))
print('Train label distribution:', y_train.value_counts(normalize=True).round(4).to_dict())
print('Test label distribution:', y_test.value_counts(normalize=True).round(4).to_dict())

## 8. Model Training

Tune at least six classifiers using cross-validation and the selected text representation.

In [ ]:
results, best = train_models(X_train, y_train, X_test, y_test, PROJECT_ROOT / 'models', best_vectorizer_name)
print('Best model:', best.name)
print('Best score:', best.best_score)
display(build_metrics_table(results))

## 9. Hyperparameter Tuning

The training script performs GridSearchCV or RandomizedSearchCV depending on the parameter grid size and records the best parameters and timing information.

In [ ]:
comparison = build_metrics_table(results)
display(comparison[['model', 'best_score', 'training_time', 'testing_time', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc']])
comparison.to_csv(METRICS_DIR / 'model_comparison.csv', index=False)

## 10. Evaluation

Inspect the final comparison table and the generated evaluation figures.

In [ ]:
display(comparison.head(10))
print('Best model selected by F1 / ROC AUC / accuracy tie-breaks:', best.name)
print('Best model metrics:', best.metrics)

## 11. Best Model

The best pipeline is persisted as `models/best_model.pkl`, and the selected text vectorizer is saved alongside it as `models/tfidf_vectorizer.pkl`.

In [ ]:
import joblib

best_model_path = PROJECT_ROOT / 'models' / 'best_model.pkl'
vectorizer_path = PROJECT_ROOT / 'models' / 'tfidf_vectorizer.pkl'
print('Best model exists:', best_model_path.exists())
print('Vectorizer exists:', vectorizer_path.exists())
if best_model_path.exists():
    print('Loaded model type:', type(joblib.load(best_model_path)).__name__)

## 12. Save Model

The training pipeline already writes all required artifacts and reports into the project folders.

In [ ]:
summary = run_training_pipeline(CSV_PATH, PROJECT_ROOT)
display(summary['metrics_table'])
print('Artifacts written to models/ and outputs/.')

## 13. Conclusion

This project provides a reusable, modular baseline for malicious URL classification. It can be extended with more models, richer reputation features, and stronger explainability tooling.